# Regularization for High Dimensional Medical Data

Goal: Work with high dimensional medical like data and apply regularization.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline



def load_dataset(name: str) -> pd.DataFrame:
    base_url = (
        "https://raw.githubusercontent.com/"
        "Jesse-Islam/QLS-MiCM_Introduction_to_supervised_machine_learning/main/Exercises/data/"
    )
    url = f"{base_url}{name}.csv"
    
    try:
        return pd.read_csv(url)
    except Exception as e:
        raise FileNotFoundError(f"Could not load dataset '{name}' from {url}") from e



def basic_train_test_split(df: pd.DataFrame, target: str, test_size: float = 0.2, random_state: int = 42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def build_preprocessor(num_features, cat_features):
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )
    return preprocessor



In [ ]:

# ===========================
# 0. Explore the Omics dataset
# ===========================

# Load high-dimensional dataset
df = load_dataset("omics")

# 0.1: Inspect the dataset
display(df.head())
print(f"Shape: {df.shape}")
print(df.describe().T.head(10))  # summarize first few numeric features

# 0.3: Visualize the target distribution (simple matplotlib)
target = "inflammation_index"
plt.figure(figsize=(6,4))
plt.hist(df[target], bins=25, color="skyblue", edgecolor="black")
plt.title(f"Distribution of {target}")
plt.xlabel(target)
plt.ylabel("Count")
plt.grid(alpha=0.3)
plt.show()

# 0.4: Pairwise correlations with the target (top 10 features)
corrs = df.corr(numeric_only=True)[target].drop(target).sort_values(ascending=False)
top_corrs = corrs.head(10)

plt.figure(figsize=(8,5))
plt.barh(top_corrs.index[::-1], top_corrs.values[::-1], color="steelblue", edgecolor="black")
plt.title(f"Top 10 features correlated with {target}")
plt.xlabel("Correlation with target")
plt.tight_layout()
plt.show()

# 0.5: Optional scatterplot for the strongest correlation
top_feature = top_corrs.index[0]
plt.figure(figsize=(5,4))
plt.scatter(df[top_feature], df[target], alpha=0.6, edgecolor="black")
plt.title(f"{target} vs {top_feature}")
plt.xlabel(top_feature)
plt.ylabel(target)
plt.grid(alpha=0.3)
plt.show()

print(f"The most correlated feature with {target} is: '{top_feature}' ({top_corrs.iloc[0]:.2f})")

X_train, X_test, y_train, y_test = basic_train_test_split(
    df, target="inflammation_index"
)

# Baseline model: plain OLS (no regularization)
base_model = LinearRegression()
base_model.fit(X_train, y_train)
base_pred = base_model.predict(X_test)
print("Baseline MSE:", mean_squared_error(y_test, base_pred))


In [ ]:

# ----------------------------------
# TODO 2.1: Ridge with cross validation (with scaling)
# ----------------------------------
ridge_pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("ridge", Ridge())
    ]
)

ridge_param_grid = {
    "ridge__alpha": [0.1, 1.0, 10.0]
}

ridge_cv = GridSearchCV(ridge_pipe, ridge_param_grid, cv=5)
ridge_cv.fit(X_train, y_train)

ridge_pred = ridge_cv.predict(X_test)
print("Ridge best alpha:", ridge_cv.best_params_)
print("Ridge MSE:", mean_squared_error(y_test, ridge_pred))

# Extract ridge coefficients (from the model inside the pipeline)
ridge_best_model = ridge_cv.best_estimator_["ridge"]
ridge_coefs = pd.Series(
    ridge_best_model.coef_,
    index=X_train.columns,
    name="ridge_coef"
)
print("\nTop 10 Ridge coefficients (by absolute value):")
display(ridge_coefs.reindex(ridge_coefs.abs().sort_values(ascending=False).head(10).index))


In [ ]:

# ----------------------------------
# TODO 2.2: Lasso with cross validation (with scaling)
# ----------------------------------
lasso_pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("lasso", Lasso(max_iter=5000))
    ]
)

lasso_param_grid = {
    "lasso__alpha": [0.001, 0.01, 0.1]
}

lasso_cv = GridSearchCV(lasso_pipe, lasso_param_grid, cv=5)
lasso_cv.fit(X_train, y_train)

lasso_pred = lasso_cv.predict(X_test)
print("Lasso best alpha:", lasso_cv.best_params_)
print("Lasso MSE:", mean_squared_error(y_test, lasso_pred))

# Extract lasso coefficients
lasso_best_model = lasso_cv.best_estimator_["lasso"]
lasso_coefs = pd.Series(
    lasso_best_model.coef_,
    index=X_train.columns,
    name="lasso_coef"
)

# Features that were selected (nonzero coefficients)
selected_features = lasso_coefs[lasso_coefs != 0]
print(f"Lasso selected {len(selected_features)} of {len(lasso_coefs)} features.")
display(selected_features.reindex(selected_features.abs().sort_values(ascending=False).head(15).index))



In [ ]:
#***EXERCISE*** fill in the pipelines


# ----------------------------------
# Helper function with scaling
# ----------------------------------
def train_regularized_model(X_train, y_train, model_type="ridge"):
    if model_type == "ridge":
        pipe = Pipeline(#?
        )
        grid = {"model__alpha": [0.1, 1.0, 10.0]}
    else:
        pipe = Pipeline(#?
        )
        grid = {"model__alpha": [0.001, 0.01, 0.1]}
    search = GridSearchCV(pipe, grid, cv=5)
    search.fit(X_train, y_train)
    return search

# usage
ridge_search = train_regularized_model(X_train, y_train, model_type="ridge")
lasso_search = train_regularized_model(X_train, y_train, model_type="lasso")

# extract selected from the helper
lasso_best = lasso_search.best_estimator_["model"]
lasso_coefs = pd.Series(lasso_best.coef_, index=X_train.columns)
selected = lasso_coefs[lasso_coefs != 0]
# display(selected)
# **EXERCISE** extract selected for ridge


### Interpretation

- Which variables have the **largest absolute coefficients** under Ridge?  
  - These are most strongly associated with SBP when all others are controlled for.

- Which variables were **set to zero** under Lasso?  
  - These are features that Lasso considered unimportant (they add little predictive value).

- Compare the selected variables to those that had small p-values in the previous statsmodels analysis.
  - Are the same variables identified as important?
  - What might explain differences between significance and predictive strength?

**Key takeaway:**  
Regularization offers a scalable alternative to classical inference.  
Where p-values test each variable individually, Lasso and Ridge automatically identify and shrink uninformative features - a practical approach when there are many correlated predictors.


In [ ]:

# **EXERCISE** What's missing here? figure it out.
# we want to perform ridge regression and lasso on the blood pressure dataset
# what happens when we increase alpha? are there issues with that?
# ==========================
# PART 2: EXERCISE on simpler BP dataset
# ==========================

# 1. Load the simpler dataset
bp_df = load_dataset("bp")

# 2. Add the interaction term
bp_df["bmi_sex_interaction"] = bp_df["bmi"] * bp_df["sex"]

# 3. Split into train and test sets
features = ["age", "bmi", "sodium", "exercise_minutes", "sex", "bmi_sex_interaction"]
X_train_bp, X_test_bp, y_train_bp, y_test_bp = basic_train_test_split(bp_df, target="sbp")

# 4. Ridge Regression (with scaling)
bp_ridge_pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("ridge", Ridge())
    ]
)
bp_ridge_grid = {"ridge__alpha": [1.0, 10.0]}
bp_ridge_cv = GridSearchCV(bp_ridge_pipe, bp_ridge_grid, cv=5)
bp_ridge_cv.fit()

bp_ridge_pred = bp_ridge_cv.predict(X_test_bp[features])
print("\n[BP] Ridge best alpha:", bp_ridge_cv.best_params_)
print("[BP] Ridge MSE:", mean_squared_error(y_test_bp, bp_ridge_pred))

bp_ridge_best = bp_ridge_cv.best_estimator_["ridge"]
bp_ridge_coefs = pd.Series(bp_ridge_best.coef_, index=features, name="ridge_coef")
print("\n[BP] Ridge coefficients:")
display(bp_ridge_coefs.round(3))

# 5. Lasso Regression (with scaling)
bp_lasso_pipe = Pipeline(
)

bp_lasso_cv = GridSearchCV(bp_lasso_pipe, bp_lasso_grid, cv=5)
bp_lasso_cv.fit(X_train_bp[features], y_train_bp)

bp_lasso_pred = bp_lasso_cv.predict(X_test_bp[features])
print("[BP] Lasso best alpha:", bp_lasso_cv.best_params_)
print("[BP] Lasso MSE:", mean_squared_error(y_test_bp, bp_lasso_pred))

bp_lasso_best = #bp_lasso_cv.best_estimator_["lasso"]
bp_lasso_coefs = pd.Series(bp_lasso_best.coef_, index=features, name="lasso_coef")
print("\n[BP] Lasso coefficients:")
display(bp_lasso_coefs.round(3))

# 6. Identify selected features (nonzero)
bp_selected = bp_lasso_coefs[bp_lasso_coefs != 0]
print(f"[BP] Lasso selected {len(bp_selected)} out of {len(bp_lasso_coefs)} features.")
display(bp_selected.reindex(bp_selected.abs().sort_values(ascending=False).index))
